# Federal Reserve Press Releases — Full-Corpus LLM Relevance Audit - v2

เป้าหมายของ Notebook นี้คือ:
1. ใช้ records ทั้งหมดจาก Clean Master โดยไม่สุ่มหรือจำกัดจำนวน
2. ส่ง records ทั้งหมดให้ LLM ช่วยจำแนกความเกี่ยวข้องกับ Gold Assistant
3. บันทึก label, confidence, evidence, factors และเหตุผล
4. สร้าง CSV ให้มนุษย์ตรวจผล LLM และเก็บ full-corpus metadata

> Cached outputs are historical. Running the classification cells processes the full Clean Master. The offline v3 revalidation section at the end still reports on preserved historical v2 results and does not make API calls.

**ไม่แก้/ลบ Clean Master** — อ่าน input แบบ read-only และเขียนผลลงโฟลเดอร์ใหม่

> Notebook นี้ใช้ **ThaiLLM** (`openthaigpt-thaillm-8b-instruct-v7.2`) ผ่าน endpoint แบบ OpenAI-compatible
> และกระจายงานไปยัง API key ทั้ง 7 ตัว (`THAILLM_API_KEY1..7` ที่โหลดจาก `.env`) แบบขนาน (parallel)
>
> ควรตรวจนโยบายข้อมูล/ค่าใช้จ่ายก่อนส่งเนื้อหาออกจากเครื่อง

## v2 Changes
- Added canonical JSON schema
- Improved deterministic validation
- Added Phase A baseline audit
- Separated v1 and v2 outputs
- Better evidence quote verification

In [1]:
from pathlib import Path
import os, json, re, time, hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import pandas as pd
import numpy as np
from IPython.display import display
from tqdm.auto import tqdm

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.6
numpy: 2.4.6


In [2]:
# ===== CONFIG =====
INPUT_PATH = Path(r"D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_clean.jsonl")
OUTPUT_DIR = Path(r"D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned_audit")
import sys
if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))
from fed_relevance_validation_v3 import append_jsonl_once, load_checkpoint, normalize_array_field, normalize_confidence, normalize_text as normalize_text_v3, parse_model_json, validate_classification_record, validate_model_response, verify_quote as verify_quote_v3

SELECTION_MODE = "full_corpus"

from dotenv import load_dotenv
load_dotenv()

THAILLM_BASE_URL = "http://thaillm.or.th/api/v1"
THAILLM_MODEL = "openthaigpt-thaillm-8b-instruct-v7.2"

THAILLM_API_KEYS = [os.getenv(f"THAILLM_API_KEY{i}") for i in range(1, 8)]
THAILLM_API_KEYS = [key for key in THAILLM_API_KEYS if key]

if not THAILLM_API_KEYS:
    raise EnvironmentError("ไม่พบ THAILLM_API_KEY1..7 ใน .env")

THAILLM_DELAY = 0.25
PROVIDER = "ThaiLLM"
MODEL = THAILLM_MODEL
TEMPERATURE = 0
MAX_TEXT_CHARS = 12000
MAX_TOKENS = 1500
MAX_RETRIES = 4

TYPE_FIELDS = ["document_type", "release_type", "category", "type", "page_type"]
DATE_FIELDS = ["published_at", "publication_date", "date", "release_date"]
TITLE_FIELDS = ["title", "headline", "name"]
URL_FIELDS = ["url", "source_url", "link"]
TEXT_FIELDS = ["clean_text", "body", "content", "text", "article_text"]

LABELS = ["DIRECT_GOLD", "MACRO_RELEVANT", "LOW_RELEVANCE", "UNCERTAIN"]

print("Input:", INPUT_PATH.resolve())
print("Output:", OUTPUT_DIR.resolve())
print("Provider:", PROVIDER, "| Model:", MODEL)
print("API keys loaded:", len(THAILLM_API_KEYS))

Input: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_clean.jsonl
Output: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned_audit
Provider: ThaiLLM | Model: openthaigpt-thaillm-8b-instruct-v7.2
API keys loaded: 7


In [3]:
# ===== LOAD CLEAN MASTER READ-ONLY =====
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"ไม่พบ Clean Master: {INPUT_PATH}")

records = []
with INPUT_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"JSONL parse error ที่บรรทัด {line_no}: {exc}") from exc
        if isinstance(obj, dict):
            records.append(obj)

df = pd.DataFrame(records)
print(f"Loaded {len(df):,} records")

Loaded 4,629 records


In [4]:
# ===== PREPARE YEAR × DOCUMENT TYPE STRATA =====
def first_nonempty(row, candidates, default=""):
    for field in candidates:
        if field in row.index:
            value = row[field]
            if pd.notna(value) and str(value).strip():
                return str(value).strip()
    return default

def parse_year(value):
    match = re.search(r"\b(19\d{2}|20\d{2})\b", str(value or ""))
    return int(match.group(1)) if match else None

def infer_document_type(row):
    supplied = first_nonempty(row, TYPE_FIELDS)
    if supplied and supplied.lower() not in {"nan", "none", "unknown", "null"}:
        return supplied[:80]
    s = (first_nonempty(row, TITLE_FIELDS) + " " + first_nonempty(row, URL_FIELDS)).lower()
    rules = [
        ("FOMC / Monetary Policy", ["fomc", "federal open market", "monetary policy", "target range for the federal funds"]),
        ("Speeches / Remarks", ["speech", "remarks", "address", "keynote"]),
        ("Testimony", ["testimony", "testifies"]),
        ("Research / Reports", ["research", "working paper", "survey", "report"]),
        ("Regulation / Supervision", ["supervision", "regulation", "banking", "capital rule", "stress test"]),
        ("Financial Stability", ["financial stability", "financial markets", "liquidity"]),
        ("Payments / Operations", ["payment", "fednow", "currency"]),
    ]
    for label, words in rules:
        if any(word in s for word in words):
            return label
    return "General / Other"

df["_title"] = df.apply(lambda r: first_nonempty(r, TITLE_FIELDS, "(untitled)"), axis=1)
df["_url"] = df.apply(lambda r: first_nonempty(r, URL_FIELDS), axis=1)
df["_date_raw"] = df.apply(lambda r: first_nonempty(r, DATE_FIELDS), axis=1)
df["_year"] = df["_date_raw"].apply(parse_year)
df["_document_type"] = df.apply(infer_document_type, axis=1)
df["_review_text"] = df.apply(lambda r: first_nonempty(r, TEXT_FIELDS), axis=1)

def is_archive_candidate(row):
    flag = str(row.get("archive_type", "")).strip().lower()
    title, url = row["_title"].lower(), row["_url"].lower()
    return flag in {"false", "0", "archive", "archive_page", "index"} or ("/archives/" in url and not row["_date_raw"]) or title == "press releases"

df["_archive_candidate"] = df.apply(is_archive_candidate, axis=1)
frame = df.loc[~df["_archive_candidate"]].copy()
frame["_year_stratum"] = frame["_year"].apply(lambda x: str(int(x)) if pd.notna(x) else "Unknown year")
frame["_type_stratum"] = frame["_document_type"].fillna("General / Other").astype(str)
frame["_stratum"] = frame["_year_stratum"] + " | " + frame["_type_stratum"]

print("Sampling frame:", len(frame))

Sampling frame: 4629


In [ ]:
# ===== FULL CORPUS (NO SAMPLING) =====
N = len(frame)
sample = frame.copy().reset_index(drop=True)
sample["record_id"] = sample.apply(
    lambda r: hashlib.sha256((r["_url"] or f"{r['_title']}|{r['_date_raw']}").encode("utf-8")).hexdigest()[:16],
    axis=1
)

if len(sample) != N:
    raise ValueError(f"Full-corpus selection mismatch: selected {len(sample)} of {N} records")
if not sample["record_id"].is_unique:
    raise ValueError("Duplicate record_id found in full Clean Master corpus")
print(f"Selected all {len(sample)} / {N} records in Clean Master; no sampling")

In [6]:
# ===== IMPORT THAILLM CLIENT =====
try:
    from openai import OpenAI
except ImportError:
    raise ImportError("ติดตั้ง openai: %pip install openai")

clients = [
    OpenAI(base_url=THAILLM_BASE_URL, api_key=key)
    for key in THAILLM_API_KEYS
]
num_keys = len(clients)

def get_llm_index(record_id):
    hash_value = int(hashlib.md5(str(record_id).encode("utf-8")).hexdigest(), 16)
    return hash_value % max(1, num_keys)

print(f"ThaiLLM clients: {num_keys} instances")

ThaiLLM clients: 7 instances


In [ ]:
# ===== PHASE A: BASELINE AUDIT REPORT =====
from datetime import datetime

print("=" * 60)
print("PHASE A: BASELINE AUDIT REPORT")
print("=" * 60)
print(f"Timestamp: {datetime.now().isoformat()}")
print()

print("1. CLEAN MASTER")
print(f"   Path: {INPUT_PATH}")
print(f"   Total records: {len(df):,}")
print()

print("2. FULL CORPUS")
print(f"   Selected records: {len(sample)}")
print(f"   Unique record_ids: {sample['record_id'].nunique()}")
print()

print("=" * 60)
print("PHASE A COMPLETE")
print("=" * 60)

In [8]:
# ===== PHASE B: CANONICAL JSON SCHEMA =====
CANONICAL_SCHEMA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "Fed Gold Relevance Classification Output",
    "type": "object",
    "required": [
        "record_id",
        "llm_primary_label",
        "llm_confidence",
        "llm_document_type",
        "llm_main_topic",
        "llm_evidence_quote",
        "evidence_quote_found_in_text",
        "llm_economic_factor",
        "llm_relevance_pathway",
        "llm_relevance_reason",
        "llm_limitations",
    ],
    "properties": {
        "record_id": {"type": "string", "minLength": 1},
        "llm_primary_label": {"type": "string", "enum": LABELS},
        "llm_confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
        "llm_document_type": {"type": "string", "minLength": 1},
        "llm_main_topic": {"type": "string", "minLength": 1},
        "llm_evidence_quote": {"type": "string"},
        "evidence_quote_found_in_text": {"type": ["boolean", "null"]},
        "llm_economic_factor": {"type": "array", "items": {"type": "string"}},
        "llm_relevance_pathway": {"type": "string"},
        "llm_relevance_reason": {"type": "string"},
        "llm_limitations": {"type": "array", "items": {"type": "string"}},
    }
}

print("Canonical schema defined")
print(f"Required fields: {len(CANONICAL_SCHEMA['required'])}")

Canonical schema defined
Required fields: 11


In [9]:
# ===== PHASE B: DETERMINISTIC VALIDATION =====
import unicodedata

def normalize_text(text):
    return normalize_text_v3(text if isinstance(text, str) else "")

def verify_quote(article_text, quote):
    result = verify_quote_v3(article_text, quote)
    return {"quote_verified": result["verified"], "verification_status": result["status"]}

def validate_against_canonical_schema(obj):
    canonical = {field: obj[field] for field in CANONICAL_SCHEMA["required"] if field in obj}
    schema_errors = validate_classification_record(canonical)
    return {"schema_valid": not schema_errors, "schema_errors": schema_errors, "quarantined": bool(schema_errors)}

def improved_validate_schema(parsed_result, body_text):
    evidence_quote = parsed_result.get("llm_evidence_quote", "")
    primary_label = parsed_result.get("llm_primary_label")
    quote_result = verify_quote(body_text, evidence_quote)
    if quote_result["verification_status"] == "EMPTY_QUOTE" and primary_label in ("LOW_RELEVANCE", "UNCERTAIN"):
        quote_result["verification_status"] = "QUOTE_NOT_REQUIRED"
    canonical = {**parsed_result, "evidence_quote_found_in_text": quote_result["quote_verified"]}
    schema_validation = validate_against_canonical_schema(canonical)
    validation = {
        "schema_valid": schema_validation["schema_valid"],
        "schema_errors": schema_validation["schema_errors"],
        "quote_verification": quote_result,
        "consistency_flags": []
    }
    
    if not schema_validation["schema_valid"]:
        validation["consistency_flags"].append("SCHEMA_INVALID")
        return validation
    
    quote_status = "EMPTY_QUOTE"
    quote_status = quote_result["verification_status"]
    if evidence_quote:
        if quote_result["quote_verified"] is not True:
            if primary_label in ("DIRECT_GOLD", "MACRO_RELEVANT"):
                validation["consistency_flags"].append("QUOTE_NOT_VERIFIED")
    
    confidence = parsed_result.get("llm_confidence")
    economic_factor = parsed_result.get("llm_economic_factor", [])
    pathway = str(parsed_result.get("llm_relevance_pathway", "")).lower()
    
    if primary_label == "DIRECT_GOLD":
        if quote_status != "EXACT_NORMALIZED_MATCH" or not any(term in evidence_quote.lower() for term in ("gold", "bullion", "precious metal")):
            validation["consistency_flags"].append("DIRECT_GOLD_REQUIRES_REVIEW")
    
    if primary_label == "MACRO_RELEVANT":
        if quote_status != "EXACT_NORMALIZED_MATCH":
            validation["consistency_flags"].append("QUOTE_NOT_VERIFIED")
        if not economic_factor or not any(term in pathway for term in ("interest rate", "real rate", "yield", "inflation", "dollar", "currency", "employment", "liquidity", "risk", "financial stress", "monetary policy")):
            validation["consistency_flags"].append("PATHWAY_UNSUPPORTED")
    
    if primary_label == "UNCERTAIN":
        validation["consistency_flags"].append("UNCERTAIN_CLASSIFICATION")
    
    if confidence is not None and confidence < 0.75:
        validation["consistency_flags"].append("LOW_CONFIDENCE")
    
    if not body_text.strip():
        validation["consistency_flags"].append("SOURCE_TEXT_MISSING")
    if primary_label == "LOW_RELEVANCE" and any(term in normalize_text(body_text).lower() for term in ("gold", "bullion", "gold market", "gold price")):
        validation["consistency_flags"].append("LABEL_EVIDENCE_CONFLICT")
    
    return validation

print("Validation functions defined")

Validation functions defined


In [10]:
# ===== LLM CLASSIFICATION PROMPT v2 + PARSER =====
SYSTEM_PROMPT = '''
FED DOCUMENT RELEVANCE AUDITOR — GOLD RESEARCH v2

1. ROLE
You are a conservative document-relevance auditor for a quantitative gold-market research corpus.
Classify only what is supported by the supplied document text and metadata.
You are NOT predicting gold prices.

2. CLASSIFICATION LABELS
Choose exactly one: DIRECT_GOLD, MACRO_RELEVANT, LOW_RELEVANCE, UNCERTAIN

DIRECT_GOLD: Use only for explicit substantive gold-market, investment, demand, supply, or product content. A general rate/inflation/USD discussion is not direct gold evidence.
MACRO_RELEVANT: Use for substantive macro, policy, currency, inflation, employment, liquidity, or risk information with an identifiable gold transmission mechanism. A generic Federal Reserve reference or routine administrative notice is insufficient.
LOW_RELEVANCE: The inspected body contains no substantive useful gold-research information or the proposed connection is speculative.
UNCERTAIN: The source is missing, incomplete, corrupted, contradictory, or otherwise insufficient.

3. EVIDENCE REQUIREMENTS
For DIRECT_GOLD or MACRO_RELEVANT: provide exact quote from article_text.
Do not fabricate quotations.

4. OUTPUT FORMAT
Return exactly one valid JSON object:
{
"llm_primary_label": "DIRECT_GOLD | MACRO_RELEVANT | LOW_RELEVANCE | UNCERTAIN",
"llm_confidence": 0.0,
"llm_document_type": "string",
"llm_main_topic": "string",
"llm_evidence_quote": "string",
"llm_economic_factor": ["string"],
"llm_relevance_pathway": "string",
"llm_relevance_reason": "string",
"llm_limitations": ["string"]
}

CRITICAL: llm_economic_factor and llm_limitations must be arrays, not strings.
Return only the JSON object, no Markdown.
'''

def parse_llm_json(raw):
    return parse_model_json(raw)

def classify_document(row, client):
    title = str(row["_title"])[:500]
    body = str(row["_review_text"] or "")[:MAX_TEXT_CHARS]
    
    user_payload = {
        "title": title,
        "publicationDate": str(row["_date_raw"]),
        "url": str(row["_url"]),
        "document_type": str(row["_document_type"]),
        "article_text": body,
    }
    
    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
        ],
    )
    raw = response.choices[0].message.content
    result = parse_llm_json(raw)
    
    primary_label = result.get("llm_primary_label")
    if primary_label not in LABELS:
        raise ValueError(f"Unknown llm_primary_label: {primary_label!r}")
    
    confidence, _, confidence_error = normalize_confidence(result.get("llm_confidence"))
    if confidence_error:
        raise ValueError(confidence_error)
    
    evidence_quote = result.get("llm_evidence_quote")
    
    economic_factor, _, factor_error = normalize_array_field(result.get("llm_economic_factor"), "llm_economic_factor")
    if factor_error:
        raise ValueError(factor_error)
    
    limitations, _, limitations_error = normalize_array_field(result.get("llm_limitations"), "llm_limitations")
    if limitations_error:
        raise ValueError(limitations_error)
    
    normalized_model = {**result, "llm_confidence": confidence, "llm_economic_factor": economic_factor, "llm_limitations": limitations}
    model_errors = validate_model_response(normalized_model)
    if model_errors:
        raise ValueError("; ".join(model_errors))
    
    return {
        "llm_primary_label": primary_label,
        "llm_confidence": confidence,
        "llm_document_type": result.get("llm_document_type"),
        "llm_main_topic": result.get("llm_main_topic"),
        "llm_evidence_quote": evidence_quote,
        "llm_economic_factor": economic_factor,
        "llm_relevance_pathway": result.get("llm_relevance_pathway"),
        "llm_relevance_reason": result.get("llm_relevance_reason"),
        "llm_limitations": limitations,
        "llm_review_required": False,
        "raw_model_response": raw,
    }

print("Prompt v2 and parser defined")

Prompt v2 and parser defined


In [ ]:
# ===== STAGE 2+3: LLM CLASSIFICATION + VALIDATION =====
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
progress_path = OUTPUT_DIR / "llm_progress_v3.jsonl"
errors_path = OUTPUT_DIR / "llm_errors_v3.jsonl"
results_by_id = {}
write_lock = Lock()

checkpoint_state = load_checkpoint(progress_path, OUTPUT_DIR / "llm_quarantine_v3.jsonl")
results_by_id.update(checkpoint_state["completed_by_id"])
completed_ids = set(results_by_id)
print("Resuming v3:", len(results_by_id), "completed records;", len(checkpoint_state["failed_by_id"]), "previous failures eligible for retry")

pending = sample.loc[~sample["record_id"].isin(results_by_id)].copy()
print("Pending:", len(pending), "| workers:", num_keys)

def process_record(row):
    record_id = row["record_id"]
    client = clients[get_llm_index(record_id)]
    result = None
    last_error = None
    
    attempts = 0
    for attempt in range(1, MAX_RETRIES + 1):
        attempts = attempt
        time.sleep(THAILLM_DELAY)
        try:
            result = classify_document(row, client)
            break
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            time.sleep(min(2 ** attempt, 20))
    
    if result is None:
        with write_lock:
            with errors_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps({"record_id": record_id, "title": row["_title"], "error": last_error, "retry_count": MAX_RETRIES, "final_status": "failed"}, ensure_ascii=False) + "\n")
        return {"record_id": record_id, "final_status": "failed", "error": last_error}
    
    body_text = str(row["_review_text"] or "")[:MAX_TEXT_CHARS]
    
    canonical_for_validation = {"record_id": record_id, **result}
    validation = improved_validate_schema(canonical_for_validation, body_text)
    
    result["validation_schema_valid"] = validation["schema_valid"]
    result["validation_schema_errors"] = validation["schema_errors"]
    result["evidence_quote_found_in_text"] = validation["quote_verification"]["quote_verified"]
    result["validation_quote_verified"] = validation["quote_verification"]["quote_verified"] if validation["quote_verification"] else None
    result["validation_quote_status"] = validation["quote_verification"]["verification_status"] if validation["quote_verification"] else None
    result["validation_consistency_flags"] = validation["consistency_flags"]
    result["llm_review_required"] = bool(not validation["schema_valid"] or validation["consistency_flags"])
    
    if validation["consistency_flags"]:
        result["llm_review_required"] = True
    
    result["input_hash"] = hashlib.sha256(body_text.encode("utf-8")).hexdigest()[:16]
    result["prompt_version"] = "fed_gold_relevance_v3"
    result["prompt_hash"] = hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest()
    result["schema_version"] = "fed-classification-v3"
    result["run_id"] = hashlib.sha256((str(INPUT_PATH.resolve()) + SELECTION_MODE + MODEL).encode("utf-8")).hexdigest()[:16]
    result["retry_count"] = attempts - 1
    result["timestamp_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    result["model"] = MODEL
    
    item = {"record_id": record_id, **result, "final_status": "completed" if validation["schema_valid"] else "quarantined"}
    with write_lock:
        if item["final_status"] == "completed":
            if not append_jsonl_once(progress_path, item, completed_ids):
                raise ValueError(f"Duplicate completed record_id on resume: {record_id}")
            results_by_id[record_id] = item
        else:
            with (OUTPUT_DIR / "llm_quarantine_v3.jsonl").open("a", encoding="utf-8") as f:
                f.write(json.dumps(item, ensure_ascii=False, allow_nan=False) + "\n")
    return item

if len(pending):
    with ThreadPoolExecutor(max_workers=num_keys) as executor:
        futures = [executor.submit(process_record, row) for _, row in pending.iterrows()]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="ThaiLLM classify v2"):
            pass

print(f"LLM pass complete. Total results={len(results_by_id)}")

In [ ]:
# ===== STAGE 1+4: MERGE + EXPORT FOR HUMAN AUDIT =====
if not results_by_id:
    print("ERROR: No LLM results. Run Cell 10 first.")
else:
    results_df = pd.DataFrame(list(results_by_id.values()))
    merged = sample.merge(results_df, on="record_id", how="left", validate="one_to_one")

    review_df = pd.DataFrame({
        "record_id": merged["record_id"],
        "published_date": merged["_date_raw"],
        "year": merged["_year_stratum"],
        "document_type": merged["_type_stratum"],
        "title": merged["_title"],
        "url": merged["_url"],
        "article_text": merged["_review_text"],
        "llm_primary_label": merged.get("llm_primary_label", pd.Series(index=merged.index, dtype=object)),
        "llm_confidence": merged.get("llm_confidence", pd.Series(index=merged.index, dtype=object)),
        "llm_evidence_quote": merged.get("llm_evidence_quote", pd.Series(index=merged.index, dtype=object)),
        "llm_economic_factor": merged.get("llm_economic_factor", pd.Series(index=merged.index, dtype=object)),
        "llm_relevance_pathway": merged.get("llm_relevance_pathway", pd.Series(index=merged.index, dtype=object)),
        "llm_relevance_reason": merged.get("llm_relevance_reason", pd.Series(index=merged.index, dtype=object)),
        "llm_limitations": merged.get("llm_limitations", pd.Series(index=merged.index, dtype=object)),
        "llm_review_required": merged.get("llm_review_required", pd.Series(index=merged.index, dtype=object)),
        "validation_schema_valid": merged.get("validation_schema_valid", pd.Series(index=merged.index, dtype=object)),
        "validation_quote_verified": merged.get("validation_quote_verified", pd.Series(index=merged.index, dtype=object)),
        "validation_quote_status": merged.get("validation_quote_status", pd.Series(index=merged.index, dtype=object)),
        "validation_consistency_flags": merged.get("validation_consistency_flags", pd.Series(index=merged.index, dtype=object)),
        "prompt_version": merged.get("prompt_version", pd.Series(index=merged.index, dtype=object)),
        "human_label": "",
        "human_reason": "",
    })

    review_path = OUTPUT_DIR / "fed_gold_relevance_llm_human_review_v3.csv"
    review_df.to_csv(review_path, index=False, encoding="utf-8-sig")
    print(f"Review CSV: {review_path}")
    print(f"Total records: {len(review_df)}")

In [ ]:
# ===== METADATA + VALIDATION SUMMARY =====
if 'review_path' not in locals():
    review_path = OUTPUT_DIR / "fed_gold_relevance_llm_human_review_v3.csv"

metadata = {
    "project": "Gold Assistant - Fed relevance audit v2",
    "input_path": str(INPUT_PATH),
    "input_record_count": int(len(df)),
    "selected_record_count": int(len(sample)),
    "source_record_count": int(len(frame)),
    "selection_mode": SELECTION_MODE,
    "sampling_applied": False,
    "model": MODEL,
    "prompt_version": "fed_gold_relevance_v3",
    "validation_enabled": True,
    "human_review_csv": str(review_path),
    "progress_file": str(progress_path),
}

with (OUTPUT_DIR / "sampling_and_llm_metadata_v3.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

if 'review_df' in locals():
    summary = review_df["llm_primary_label"].value_counts(dropna=False).rename_axis("llm_primary_label").reset_index(name="count")
    summary["share"] = summary["count"] / len(review_df)
    summary.to_csv(OUTPUT_DIR / "llm_label_summary_v3.csv", index=False, encoding="utf-8-sig")
    display(summary)
    
    validation_summary = {
        "total_records": len(review_df),
        "schema_valid": int(review_df["validation_schema_valid"].sum()),
        "quote_verified": int(review_df["validation_quote_verified"].sum()),
        "review_required": int(review_df["llm_review_required"].sum()),
    }
    with (OUTPUT_DIR / "validation_summary_v3.json").open("w", encoding="utf-8") as f:
        json.dump(validation_summary, f, ensure_ascii=False, indent=2)
    print("\nValidation summary:")
    print(json.dumps(validation_summary, indent=2))
else:
    print("Run Cell 11 first to generate review data.")

print("Metadata saved (v2).")

## Run Instructions

Run cells in order:
1. Cells 0-6: Setup and full-corpus selection
2. Cell 7: Phase A baseline audit
3. Cell 8: Phase B canonical schema
4. Cell 9: Phase B validation functions
5. Cell 10: Prompt v2 + parser
6. Cell 11: LLM classification + validation (long running, calls ThaiLLM API)
7. Cell 12: Merge + export for human audit
8. Cell 13: Metadata + validation summary

## Output Files
- Existing `llm_progress_v2.jsonl` and all v2 reports are preserved as audit inputs.
- `llm_progress_v3.jsonl`: successful new model results only; raw responses and run metadata are retained.
- `llm_errors_v3.jsonl`: failed requests, retry counts, and errors; failures are never recorded as completed labels.
- `llm_quarantine_v3.jsonl`: malformed or schema-invalid model results.
- `fed_gold_relevance_llm_human_review_v3.csv`: new human review export.
- `sampling_and_llm_metadata_v3.json`, `llm_label_summary_v3.csv`, and `validation_summary_v3.json`: new run reports.
- The offline v3 cell revalidates the preserved v2 results and writes canonical, review, transition, and RAG outputs without model API calls.

## v3 Offline Revalidation (no model API calls)

Run the next cell to revalidate the existing v2 JSONL against the Clean Master. It writes versioned v3 artifacts and does not overwrite `llm_progress_v2.jsonl`, Clean Master, or prior v2 exports. The shared validator injects the trusted sample `record_id` before canonical validation, verifies quotes against the uniquely joined source body, and computes separate review and RAG eligibility decisions.

In [17]:
from pathlib import Path
import sys

WORKSPACE = Path(r"D:\Project_Github\gold assisstant")
AUDIT_DIR = WORKSPACE / "dataset" / "news" / "fed" / "fed_press_releases_cleaned_audit"
CLEAN_MASTER_PATH = WORKSPACE / "dataset" / "news" / "fed" / "fed_press_releases_cleaned" / "fed_press_releases_clean.jsonl"
V2_PROGRESS_PATH = AUDIT_DIR / "llm_progress_v2.jsonl"

if str(AUDIT_DIR) not in sys.path:
    sys.path.insert(0, str(AUDIT_DIR))
from fed_relevance_validation_v3 import run_audit

v3_report = run_audit(CLEAN_MASTER_PATH, V2_PROGRESS_PATH, AUDIT_DIR)
print(v3_report["after"])


{'total_parsed_records': 200, 'processed_records': 200, 'accepted_canonical_output_records': 200, 'invalid_json_lines': 0, 'json_parse_pass_rate': 1.0, 'canonical_schema_valid': 200, 'canonical_schema_invalid': 0, 'canonical_schema_pass_rate': 1.0, 'quarantined_count': 0, 'label_distribution': {'LOW_RELEVANCE': 149, 'MACRO_RELEVANT': 48, 'UNCERTAIN': 3}, 'source_matched_count': 200, 'source_match_rate': 1.0, 'quote_status_distribution': {'EXACT_NORMALIZED_MATCH': 61, 'QUOTE_NOT_FOUND': 10, 'QUOTE_NOT_REQUIRED': 129}, 'required_quote_records': 48, 'required_quote_exact_matches': 44, 'required_quote_failures': 4, 'required_quote_failure_rate': 0.08333333333333333, 'empty_quote_count': 0, 'quote_not_required_count': 129, 'labels_not_requiring_quote_count': 152, 'review_required_count': 29, 'review_rate': 0.145, 'review_reason_counts': {'LABEL_EVIDENCE_CONFLICT': 20, 'LOW_CONFIDENCE': 3, 'PATHWAY_UNSUPPORTED': 18, 'QUOTE_NOT_VERIFIED': 4, 'UNCERTAIN_CLASSIFICATION': 3}, 'review_reasons_by_

## v4 Source-level Relevance Audit

This section runs offline against all new full-corpus v3 classifications and Clean Master. Run it after the full LLM classification cell finishes. It does not call ThaiLLM or change v3/older outputs. Rule-derived scores are triage proposals, not human-verified truth. Initial Core RAG export stays empty until every selected record has an explicit human decision and reviewer note.

In [ ]:
from fed_relevance_audit_v4 import run_v4

V3_CLASSIFICATIONS_PATH = AUDIT_DIR / "llm_progress_v3.jsonl"
v4_report = run_v4(CLEAN_MASTER_PATH, V3_CLASSIFICATIONS_PATH, AUDIT_DIR)
print(v4_report["audit_coverage"])
print(v4_report["rag_comparison"])

# After reviewer decisions are saved in the CSV, pass its path as the fourth argument:
# v4_report = run_v4(CLEAN_MASTER_PATH, V3_CLASSIFICATIONS_PATH, AUDIT_DIR, AUDIT_DIR / "fed_gold_human_review_v4.csv")
